In [1]:
import os
import re
import math
import copy
import numpy as np
import pandas as pd
import time

import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import TensorDataset, DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim

import gensim.downloader as api

import nltk
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

In [2]:
DATA_FOLDER = r"data"
DATASET_FILE = "en-US.jsonl"

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
df = pd.read_json(os.path.join(DATA_FOLDER,DATASET_FILE), lines=True)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst example:")
print(df.iloc[0])

Dataset shape: (16521, 8)

Columns:
['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id']

First example:
id                                                           0
locale                                                   en-US
partition                                                 test
scenario                                                 alarm
intent                                               alarm_set
utt                            wake me up at five am this week
annot_utt    wake me up at [time : five am] [date : this week]
worker_id                                                    1
Name: 0, dtype: object


In [4]:
# Check the number of examples in each partition
print("Partition distribution:")
print(df["partition"].value_counts())

# Verify the official dataset split sizes
assert (df["partition"] == "train").sum() == 11514
assert (df["partition"] == "dev").sum() == 2033
assert (df["partition"] == "test").sum() == 2974

# Check the number of unique intents
print("\nNumber of unique intents:")
print(df["intent"].nunique())

# Check the number of examples per intent
print("\nExamples per intent:")
display(df["intent"].value_counts().describe())

# Check for missing values
print("\nMissing values:")
print(df[["utt", "annot_utt", "intent", "partition"]].isnull().sum())

# Check whether the dataset contains only en-US
print("\nLocales:")
print(df["locale"].value_counts())

Partition distribution:
partition
train    11514
test      2974
dev       2033
Name: count, dtype: int64

Number of unique intents:
60

Examples per intent:


count      60.000000
mean      275.350000
std       253.817352
min         6.000000
25%       114.750000
50%       183.000000
75%       315.500000
max      1150.000000
Name: count, dtype: float64


Missing values:
utt          0
annot_utt    0
intent       0
partition    0
dtype: int64

Locales:
locale
en-US    16521
Name: count, dtype: int64


In [5]:
train_df = df[df["partition"] == "train"].copy()
val_df = df[df["partition"] == "dev"].copy()
test_df = df[df["partition"] == "test"].copy()

In [6]:
x_train = train_df["utt"].tolist()
y_train = train_df["intent"].tolist()
z_train = train_df["annot_utt"].tolist()

x_val = val_df["utt"].tolist()
y_val = val_df["intent"].tolist()
z_val = val_df["annot_utt"].tolist()

x_test = test_df["utt"].tolist()
y_test = test_df["intent"].tolist()
z_test = test_df["annot_utt"].tolist()

In [7]:
print("Training examples:", len(x_train))
print("Validation examples:", len(x_val))
print("Test examples:", len(x_test))

print("\nExample utterance:", x_train[0])
print("Example intent:", y_train[0])
print("Example annotated utterance:", z_train[0])

Training examples: 11514
Validation examples: 2033
Test examples: 2974

Example utterance: wake me up at nine am on friday
Example intent: alarm_set
Example annotated utterance: wake me up at [time : nine am] on [date : friday]


In [8]:
glove = api.load("glove-wiki-gigaword-100")

In [9]:
# Check vocabulary size
print("Vocabulary size:", len(glove.index_to_key))

# Check embedding dimension
print("Embedding dimension:", glove.vector_size)

# Get the vector for a word
print("Vector for 'music':")
print(glove["music"])

Vocabulary size: 400000
Embedding dimension: 100
Vector for 'music':
[ 1.2705e-01  3.7253e-01  2.3076e-01 -1.0727e-01  1.4273e+00  7.7990e-01
 -5.5009e-02 -1.8909e-01 -4.2828e-01  2.8443e-01  1.1920e-01 -6.1978e-01
 -3.9817e-01  1.9127e-01 -3.2098e-01 -2.4116e-01  6.0927e-01 -3.5765e-01
  2.0159e-01  9.1107e-01 -4.8977e-01 -2.8970e-01  1.0098e+00 -6.6149e-01
  7.0414e-02  1.3353e-01  4.2234e-01 -8.0188e-02  1.0722e+00  2.6426e-01
 -7.3982e-01  6.4958e-01 -4.8367e-01  8.5837e-01 -3.9008e-01 -1.3020e-01
  2.2331e-01  2.0818e-01 -4.7260e-01 -1.7275e+00  3.7209e-01  5.5544e-01
 -8.9025e-01  8.7442e-01 -2.9063e-01 -7.1175e-01 -3.3437e-01 -4.1141e-01
  9.9049e-01 -4.6207e-01 -2.5777e-01  4.9513e-02 -2.2242e-01 -3.3757e-01
 -1.5677e-01 -2.9576e+00 -5.8878e-01  2.2432e-01  1.6375e+00  1.0263e+00
  8.9573e-01  1.3718e+00 -3.7709e-01 -8.2095e-02  1.6339e-01 -2.5294e-01
  1.0987e+00 -2.0032e-03  7.9235e-01 -2.9403e-01 -2.6906e-01  2.3123e-01
  3.3826e-01  1.7150e-01 -3.5479e-01 -1.3497e-01  6.441

In [10]:
token_pattern = re.compile(r'\w+|[^\w\s]')

def tokenize_with_offsets(text):
    matches = token_pattern.finditer(text)
    return [(m.group().lower(), m.start(), m.end()) for m in matches]

def tokenize(text):
    return [token for token, _, _ in tokenize_with_offsets(text)]

In [11]:
def check_glove_coverage(texts, glove):
    all_tokens = [
        token
        for text in texts
        for token in tokenize(text)
    ]

    unique_tokens = set(all_tokens)
    known_tokens = {token for token in unique_tokens if token in glove}

    total_tokens = len(all_tokens)
    total_unique = len(unique_tokens)
    known_count = sum(token in glove for token in all_tokens)

    print(f"Total tokens: {total_tokens:,}")
    print(f"Unique words: {total_unique:,}")
    print(f"Unique words covered: {len(known_tokens):,}")
    print(f"Unique words OOV: {total_unique - len(known_tokens):,}")

    print(f"\nUnique-word coverage: {len(known_tokens) / total_unique * 100:.2f}%")
    print(f"Token coverage: {known_count / total_tokens * 100:.2f}%")

    oov_words = sorted(unique_tokens - known_tokens)
    print("\nFirst 30 OOV words:")
    print(oov_words[:30])

In [12]:
# Check each split
print("TRAINING SET")
check_glove_coverage(x_train, glove)

print("\nVALIDATION SET")
check_glove_coverage(x_val, glove)

print("\nTEST SET")
check_glove_coverage(x_test, glove)

TRAINING SET
Total tokens: 83,008
Unique words: 5,144
Unique words covered: 4,963
Unique words OOV: 181

Unique-word coverage: 96.48%
Token coverage: 99.67%

First 30 OOV words:
['altucher', 'apahelp', 'apig', 'applebees', 'badpho', 'bardonia', 'bbcnews', 'beanos', 'beiber', 'beibers', 'bestbuy', 'blahblah', 'bobburgers', 'brexit', 'brightener', 'btaxes', 'burmuda', 'calone', 'capabara', 'capuccino', 'channin', 'chhana', 'cnncom', 'comcastcom', 'comcasts', 'companyname', 'dalls', 'dancom', 'dbell', 'ddlj']

VALIDATION SET
Total tokens: 14,522
Unique words: 2,088
Unique words covered: 2,052
Unique words OOV: 36

Unique-word coverage: 98.28%
Token coverage: 99.70%

First 30 OOV words:
['alzacz', 'arrahman', 'ashmit', 'bakey', 'brexit', 'burmuda', 'capricorns', 'culotte', 'demonetisation', 'dooa', 'eggroll', 'enalen', 'erosser', 'finlee', 'gavenda', 'gbaby', 'harrypotter', 'iliza', 'metion', 'obaba', 'orevena', 'palhunik', 'phillyx', 'reveca', 'rihana', 'schleshinger', 'swatishs', 'tiangl

In [13]:
# Special token IDs
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

PAD_ID = 0
UNK_ID = 1

# Tokenize the training utterances
tokenized_x_train = [tokenize(text) for text in x_train]

# Build vocabulary using the training set
vocab_words = sorted({
    token
    for tokens in tokenized_x_train
    for token in tokens
})

# Create word-to-ID mapping
word_to_id = {
    PAD_TOKEN: PAD_ID,
    UNK_TOKEN: UNK_ID,
}

for word in vocab_words:
    if word not in word_to_id:
        word_to_id[word] = len(word_to_id)

# Create reverse mapping
id_to_word = {idx: word for word, idx in word_to_id.items()}

print("Training vocabulary size (incl. PAD/UNK):", len(word_to_id))
print("Training tokens missing from GloVe:", sum(word not in glove for word in vocab_words))

Training vocabulary size (incl. PAD/UNK): 5146
Training tokens missing from GloVe: 181


In [14]:
USE_LEMMA_FALLBACK = False

lemmatizer = WordNetLemmatizer()

def lemmatize_token(token):
    # Try the noun form first (plurals), then the verb form (tenses)
    lemma = lemmatizer.lemmatize(token, pos='n')
    if lemma == token:
        lemma = lemmatizer.lemmatize(token, pos='v')
    return lemma

def token_to_id(token, word_to_id, use_lemma):
    # Known words keep their own ID; only unseen words fall back to their lemma
    if token in word_to_id:
        return word_to_id[token]
    if use_lemma:
        lemma = lemmatize_token(token)
        if lemma in word_to_id:
            return word_to_id[lemma]
    return UNK_ID

In [15]:
embedding_dim = glove.vector_size
vocab_size = len(word_to_id)

rng = np.random.default_rng(SEED)

embedding_matrix = rng.normal(
    0, 0.1, (vocab_size, embedding_dim)
).astype(np.float32)

embedding_matrix[PAD_ID] = 0

for word, idx in word_to_id.items():
    if word in glove:
        embedding_matrix[idx] = glove[word]

print("Embedding matrix shape:", embedding_matrix.shape)
print("Padding vector:", embedding_matrix[PAD_ID])
print("Unknown vector shape:", embedding_matrix[UNK_ID].shape)
print("Words missing from GloVe:",
      sum(word not in glove for word in vocab_words))


Embedding matrix shape: (5146, 100)
Padding vector: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0.]
Unknown vector shape: (100,)
Words missing from GloVe: 181


In [16]:
max_train_len = max(len(tokens) for tokens in tokenized_x_train)
max_len = max(64, max_train_len)
print("Longest training sentence:", max_train_len)
print("Padding length:", max_len)

Longest training sentence: 35
Padding length: 64


In [17]:
intents = sorted(set(y_train))

intent_to_id = {
    intent: idx
    for idx, intent in enumerate(intents)
}

id_to_intent = {
    idx: intent
    for intent, idx in intent_to_id.items()
}

num_classes = len(intent_to_id)

print("Number of intent classes:", num_classes)
print("\nExample mappings:")
for intent in intents[:5]:
    print(f"{intent} -> {intent_to_id[intent]}")

Number of intent classes: 60

Example mappings:
alarm_query -> 0
alarm_remove -> 1
alarm_set -> 2
audio_volume_down -> 3
audio_volume_mute -> 4


In [18]:
slot_pattern = re.compile(r'\[\s*([^:\[\]]+?)\s*:\s*([^\[\]]*?)\]')
SLOT_PAD_ID = -100  

def parse_annotated_utterance(utt, annot_utt):
    spans = []
    plain_text = ''
    last_end = 0

    for match in slot_pattern.finditer(annot_utt):
        plain_text += annot_utt[last_end:match.start()]
        slot_type = match.group(1).strip()
        slot_value = match.group(2)
        start = len(plain_text)
        plain_text += slot_value
        spans.append({
            'type': slot_type,
            'start': start,
            'end': len(plain_text),
            'text': slot_value
        })
        last_end = match.end()

    plain_text += annot_utt[last_end:]
    if plain_text != utt:
        raise ValueError(f"Slot annotation does not match utterance: {utt!r}")

    return spans


In [19]:
def make_aligned_tokens(utt, spans):
    tags = []
    offsets = []
    prev_slot = None

    for word, start, end in tokenize_with_offsets(utt):
        current_slot = None
        for i, span in enumerate(spans):
            if start < span['end'] and end > span['start']:
                if start < span['start'] or end > span['end']:
                    raise ValueError(f"A token cuts across a slot boundary: {utt!r}")
                if current_slot is not None:
                    raise ValueError(f"A token overlaps two slots: {utt!r}")
                current_slot = i

        if current_slot is None:
            tags.append('O')
        else:
            prefix = 'I' if current_slot == prev_slot else 'B'
            tags.append(f"{prefix}-{spans[current_slot]['type']}")

        offsets.append((start, end))
        prev_slot = current_slot

    for span in spans:
        if not any(s >= span['start'] and e <= span['end'] for s, e in offsets):
            raise ValueError(f"Could not align slot {span['text']!r} in {utt!r}")

    return tags, offsets


In [20]:
def prepare_joint_records(split_df):
    records = []
    
    for row_index, row in split_df.iterrows():
        text = row['utt']
        spans = parse_annotated_utterance(text, row['annot_utt'])
        tags, offsets = make_aligned_tokens(text, spans)
        tokens = tokenize(text)

        if len(tokens) != len(tags):
            raise ValueError(f"Token and slot lengths differ at row {row_index}")

        records.append({
            'source_row': int(row_index),
            'utt': text,
            'tokens': tokens,
            'offsets': offsets,
            'gold_spans': spans,
            'slot_tags': tags,
            'intent': row['intent']
        })
    return records


In [21]:
train_records = prepare_joint_records(train_df)
val_records = prepare_joint_records(val_df)
test_records = prepare_joint_records(test_df)

training_slot_types = sorted({span['type'] for row in train_records for span in row['gold_spans']})
slot_tags = ['O']

for slot_type in training_slot_types:
    slot_tags.extend([f'B-{slot_type}', f'I-{slot_type}'])

slot_to_id = {tag: i for i, tag in enumerate(slot_tags)}
id_to_slot = {i: tag for tag, i in slot_to_id.items()}
num_slot_labels = len(slot_to_id)

print('Joint dataset sizes:', len(train_records), len(val_records), len(test_records))
print('Slot types in training:', len(training_slot_types))
print('BIO labels:', num_slot_labels)
print('Example:', train_records[0]['slot_tags'])

Joint dataset sizes: 11514 2033 2974
Slot types in training: 55
BIO labels: 111
Example: ['O', 'O', 'O', 'O', 'B-time', 'I-time', 'O', 'B-date']


In [22]:
class JointNLUDataset(Dataset):
    def __init__(self, records, word_to_id, intent_to_id, 
                 slot_to_id, max_len, use_lemma=False):
        self.records = records
        self.input_ids = []
        self.attention_masks = []
        self.intent_ids = []
        self.slot_ids = []

        for row in records:
            words = [token_to_id(word, word_to_id, use_lemma) for word in row['tokens']]
            slots = [slot_to_id[tag] for tag in row['slot_tags']]
            if len(words) != len(slots):
                raise ValueError('Token and slot label counts do not match')
            if len(words) > max_len:
                raise ValueError(f"Sentence has {len(words)} tokens, limit is {max_len}")

            padding = max_len - len(words)
            self.input_ids.append(torch.tensor(words + [PAD_ID] * padding))
            self.attention_masks.append(torch.tensor([1] * len(words) + [0] * padding, 
                                                     dtype=torch.bool))
            self.intent_ids.append(torch.tensor(intent_to_id[row['intent']]))
            self.slot_ids.append(torch.tensor(slots + [SLOT_PAD_ID] * padding))

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        return {
            'input_ids': self.input_ids[index],
            'attention_mask': self.attention_masks[index],
            'intent_labels': self.intent_ids[index],
            'slot_labels': self.slot_ids[index],
            'utt': row['utt'],
            'offsets': row['offsets'],
            'gold_spans': row['gold_spans'],
            'source_row': row['source_row']
        }

In [23]:
def joint_collate_fn(samples):
    batch = {}
    for key in ('input_ids', 'attention_mask', 'intent_labels', 'slot_labels'):
        batch[key] = torch.stack([item[key] for item in samples])
    for key in ('utt', 'offsets', 'gold_spans', 'source_row'):
        batch[key] = [item[key] for item in samples]
    return batch

In [24]:
train_joint_dataset = JointNLUDataset(train_records, word_to_id, intent_to_id, slot_to_id, max_len, use_lemma=USE_LEMMA_FALLBACK)
val_joint_dataset = JointNLUDataset(val_records, word_to_id, intent_to_id, slot_to_id, max_len, use_lemma=USE_LEMMA_FALLBACK)
test_joint_dataset = JointNLUDataset(test_records, word_to_id, intent_to_id, slot_to_id, max_len, use_lemma=USE_LEMMA_FALLBACK)


In [25]:
batch_size = 64

train_joint_loader = DataLoader(train_joint_dataset, 
                                batch_size=batch_size, 
                                shuffle=True,
                                collate_fn=joint_collate_fn,
                                generator=torch.Generator().manual_seed(SEED))

val_joint_loader = DataLoader(val_joint_dataset, 
                              batch_size=batch_size, 
                              shuffle=False,
                              collate_fn=joint_collate_fn)

test_joint_loader = DataLoader(test_joint_dataset, 
                               batch_size=batch_size, 
                               shuffle=False,
                               collate_fn=joint_collate_fn)

batch = next(iter(train_joint_loader))
print('Input IDs:', batch['input_ids'].shape)
print('Intent labels:', batch['intent_labels'].shape)
print('Slot labels:', batch['slot_labels'].shape)
print('Attention mask:', batch['attention_mask'].shape)

assert batch['input_ids'].shape == batch['slot_labels'].shape
assert torch.equal(batch['slot_labels'] == SLOT_PAD_ID, ~batch['attention_mask'])

Input IDs: torch.Size([64, 64])
Intent labels: torch.Size([64])
Slot labels: torch.Size([64, 64])
Attention mask: torch.Size([64, 64])


In [26]:
def build_vocab(records):
    # Vocabulary from the given training records only
    vocab = {PAD_TOKEN: PAD_ID, UNK_TOKEN: UNK_ID}
    for word in sorted({t for row in records for t in row['tokens']}):
        vocab[word] = len(vocab)
    return vocab

def build_embedding_matrix(vocab):
    matrix_rng = np.random.default_rng(SEED)
    matrix = matrix_rng.normal(0, 0.1, (len(vocab), embedding_dim)).astype(np.float32)
    matrix[PAD_ID] = 0
    for word, idx in vocab.items():
        if word in glove:
            matrix[idx] = glove[word]
    return matrix

def make_loader(records, vocab, use_lemma=False, shuffle=False):
    dataset = JointNLUDataset(records, vocab, intent_to_id, slot_to_id, max_len, use_lemma=use_lemma)
    generator = torch.Generator().manual_seed(SEED) if shuffle else None
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      collate_fn=joint_collate_fn, generator=generator)

def lemmatize_records(records):
    # Lemmatize model input tokens only; utt, offsets and gold spans stay original
    return [{**row, 'tokens': [lemmatize_token(t) for t in row['tokens']]} for row in records]

# The helpers must reproduce the main pipeline exactly
assert build_vocab(train_records) == word_to_id
assert np.array_equal(build_embedding_matrix(word_to_id), embedding_matrix)
print("Helpers reproduce the main vocabulary and embedding matrix")

Helpers reproduce the main vocabulary and embedding matrix


In [27]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2)
            * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        # x shape: (batch_size, sequence_length, d_model)
        return x + self.pe[:, :x.size(1), :]

class IntentTransformer(nn.Module):
    def __init__(
        self,
        embedding_matrix,
        num_classes,
        nhead=4,
        num_layers=2,
        dim_feedforward=256,
        dropout=0.1,
        pad_id=PAD_ID
    ):
        super().__init__()

        embedding_matrix = torch.tensor(
            embedding_matrix, dtype=torch.float32
        )

        vocab_size, d_model = embedding_matrix.shape

        # Pretrained GloVe embeddings (frozen)
        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix,
            freeze=True,
            padding_idx=pad_id
        )

        self.positional_encoding = PositionalEncoding(d_model)

        # Transformer encoder trained from scratch
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_classes)

        self.pad_id = pad_id

    def forward(self, x):
        # x shape: (batch_size, sequence_length)

        padding_mask = x == self.pad_id

        # Convert token IDs into word vectors
        x = self.embedding(x)

        # Add word-order information
        x = self.positional_encoding(x)

        # Contextualize each token; ignore padding as attention keys
        x = self.transformer_encoder(
            x,
            src_key_padding_mask=padding_mask
        )

        # Mean pooling over real tokens only
        real_tokens = (~padding_mask).unsqueeze(-1).float()
        x = (x * real_tokens).sum(dim=1)
        x = x / real_tokens.sum(dim=1).clamp(min=1.0)

        # Predict intent logits
        x = self.dropout(x)
        logits = self.classifier(x)

        return logits

In [28]:
MODEL_CONFIG = dict(nhead=4, num_layers=4, dim_feedforward=512, dropout=0.3)

def make_model(embedding_matrix):
    # Same seed for every variant so all models start from the same weights
    torch.manual_seed(SEED)
    return IntentTransformer(embedding_matrix=embedding_matrix, num_classes=num_classes,
                             pad_id=PAD_ID, **MODEL_CONFIG)

model = make_model(embedding_matrix)
print(model)

IntentTransformer(
  (embedding): Embedding(5146, 100, padding_idx=0)
  (positional_encoding): PositionalEncoding()
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=100, out_features=100, bias=True)
        )
        (linear1): Linear(in_features=100, out_features=512, bias=True)
        (dropout): Dropout(p=0.3, inplace=False)
        (linear2): Linear(in_features=512, out_features=100, bias=True)
        (norm1): LayerNorm((100,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((100,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.3, inplace=False)
        (dropout2): Dropout(p=0.3, inplace=False)
      )
    )
  )
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=100, out_features=60, bias=True)
)


In [29]:
model.eval()

with torch.no_grad():
    logits = model(batch['input_ids'])

print("Logits shape:", logits.shape)

Logits shape: torch.Size([64, 60])


c:\Users\ZX\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


In [30]:
model = model.to(device)

# Loss function for multi-class classification
criterion = nn.CrossEntropyLoss()

print("Device:", device)
print("Trainable parameters:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))
print("Total parameters:", sum(
    p.numel() for p in model.parameters()
))

Device: cuda
Trainable parameters: 581308
Total parameters: 1095908


In [31]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for batch in train_loader:
        x_batch = batch['input_ids'].to(device)
        y_batch = batch['intent_labels'].to(device)

        # Clear gradients from the previous batch
        optimizer.zero_grad()

        # Forward pass
        logits = model(x_batch)
        loss = criterion(logits, y_batch)

        # Backpropagation and parameter update
        loss.backward()
        optimizer.step()

        # Track loss and accuracy
        total_loss += loss.item() * x_batch.size(0)
        predictions = logits.argmax(dim=1)
        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

In [32]:
def evaluate(model, data_loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in data_loader:
            x_batch = batch['input_ids'].to(device)
            y_batch = batch['intent_labels'].to(device)

            # Forward pass only
            logits = model(x_batch)
            loss = criterion(logits, y_batch)

            # Track loss and accuracy
            total_loss += loss.item() * x_batch.size(0)
            predictions = logits.argmax(dim=1)
            correct += (predictions == y_batch).sum().item()
            total += y_batch.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

In [33]:
NUM_EPOCHS = 50

def run_experiment(embedding_matrix, train_loader, val_loaders, prefix='intent'):
    # Train one model; track the best epoch separately for each validation encoding
    exp_model = make_model(embedding_matrix).to(device)
    exp_optimizer = optim.AdamW(filter(lambda p: p.requires_grad, exp_model.parameters()),
                                lr=1e-3, weight_decay=1e-4)
    best = {label: {'acc': -1.0, 'epoch': None, 'state': None} for label in val_loaders}
    history = []

    for epoch in range(1, NUM_EPOCHS + 1):
        start = time.time()
        train_loss, train_acc = train_one_epoch(exp_model, train_loader, criterion, exp_optimizer, device)
        row = {'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc}

        for label, loader in val_loaders.items():
            val_loss, val_acc = evaluate(exp_model, loader, criterion, device)
            row[f'{label} val_loss'] = val_loss
            row[f'{label} val_acc'] = val_acc
            if val_acc > best[label]['acc']:
                best[label] = {'acc': val_acc, 'epoch': epoch,
                               'state': copy.deepcopy(exp_model.state_dict())}

        row['seconds'] = time.time() - start
        history.append(row)
        val_text = " | ".join(f"{label} {row[f'{label} val_acc'] * 100:.2f}%" for label in val_loaders)
        print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | Train Loss {train_loss:.4f} Acc {train_acc:.4f} | "
              f"{val_text} | {row['seconds']:.1f}s")

    results = {}
    for label, info in best.items():
        filename = f"{prefix}_{label.lower().replace(' ', '_')}_best.pt"
        torch.save(info['state'], filename)
        results[label] = {'Best epoch': info['epoch'],
                          'Best intent accuracy (%)': round(100 * info['acc'], 2),
                          'Checkpoint': filename}
        print(f"{label}: best val {info['acc'] * 100:.2f}% at epoch {info['epoch']} -> {filename}")
    return exp_model, history, results

all_results = {}

In [34]:
# Training inputs are identical with and without the fallback, so one model serves both
train_with_lemma = JointNLUDataset(train_records, word_to_id, intent_to_id, slot_to_id, max_len, use_lemma=True)
assert all(torch.equal(a, b) for a, b in zip(train_joint_dataset.input_ids, train_with_lemma.input_ids))
print("Verified: training token IDs are identical with and without the OOV fallback")

rescued = sum(t not in word_to_id and lemmatize_token(t) in word_to_id
              for row in val_records for t in row['tokens'])
print("Unseen validation tokens rescued by lemma fallback:", rescued)

base_model, base_history, base_results = run_experiment(
    embedding_matrix,
    make_loader(train_records, word_to_id, shuffle=True),
    {'Without fallback': make_loader(val_records, word_to_id, use_lemma=False),
     'With fallback': make_loader(val_records, word_to_id, use_lemma=True)},
)
all_results.update(base_results)

Verified: training token IDs are identical with and without the OOV fallback
Unseen validation tokens rescued by lemma fallback: 62
Epoch 01/50 | Train Loss 2.4378 Acc 0.3903 | Without fallback 65.17% | With fallback 65.17% | 3.2s
Epoch 02/50 | Train Loss 1.2292 Acc 0.6682 | Without fallback 74.23% | With fallback 74.27% | 3.2s
Epoch 03/50 | Train Loss 0.9778 Acc 0.7306 | Without fallback 76.00% | With fallback 75.90% | 2.3s
Epoch 04/50 | Train Loss 0.8143 Acc 0.7752 | Without fallback 77.57% | With fallback 77.72% | 2.3s
Epoch 05/50 | Train Loss 0.7401 Acc 0.7917 | Without fallback 79.98% | With fallback 80.03% | 2.3s
Epoch 06/50 | Train Loss 0.6586 Acc 0.8111 | Without fallback 81.06% | With fallback 81.21% | 2.3s
Epoch 07/50 | Train Loss 0.6160 Acc 0.8253 | Without fallback 81.80% | With fallback 81.85% | 2.3s
Epoch 08/50 | Train Loss 0.5526 Acc 0.8359 | Without fallback 80.08% | With fallback 80.13% | 2.2s
Epoch 09/50 | Train Loss 0.5334 Acc 0.8438 | Without fallback 81.65% | With 

In [35]:
full_train_records = lemmatize_records(train_records)
full_val_records = lemmatize_records(val_records)
full_word_to_id = build_vocab(full_train_records)

print("Full-lemma vocabulary size:", len(full_word_to_id), "| original:", len(word_to_id))
print("Example:", train_records[0]['tokens'], "->", full_train_records[0]['tokens'])

full_model, full_history, full_results = run_experiment(
    build_embedding_matrix(full_word_to_id),
    make_loader(full_train_records, full_word_to_id, shuffle=True),
    {'Full lemmatization': make_loader(full_val_records, full_word_to_id)},
)
all_results.update(full_results)

Full-lemma vocabulary size: 4348 | original: 5146
Example: ['wake', 'me', 'up', 'at', 'nine', 'am', 'on', 'friday'] -> ['wake', 'me', 'up', 'at', 'nine', 'be', 'on', 'friday']
Epoch 01/50 | Train Loss 2.4677 Acc 0.3809 | Full lemmatization 62.52% | 2.3s
Epoch 02/50 | Train Loss 1.2582 Acc 0.6602 | Full lemmatization 74.03% | 2.2s
Epoch 03/50 | Train Loss 0.9965 Acc 0.7215 | Full lemmatization 75.85% | 2.2s
Epoch 04/50 | Train Loss 0.8378 Acc 0.7681 | Full lemmatization 77.37% | 2.3s
Epoch 05/50 | Train Loss 0.7541 Acc 0.7877 | Full lemmatization 80.62% | 2.2s
Epoch 06/50 | Train Loss 0.6731 Acc 0.8095 | Full lemmatization 79.69% | 2.3s
Epoch 07/50 | Train Loss 0.6196 Acc 0.8244 | Full lemmatization 81.55% | 2.1s
Epoch 08/50 | Train Loss 0.5840 Acc 0.8305 | Full lemmatization 81.60% | 2.2s
Epoch 09/50 | Train Loss 0.5510 Acc 0.8410 | Full lemmatization 82.14% | 2.2s
Epoch 10/50 | Train Loss 0.5313 Acc 0.8466 | Full lemmatization 83.87% | 2.2s
Epoch 11/50 | Train Loss 0.4960 Acc 0.8564 |

In [36]:
comparison = pd.DataFrame.from_dict(all_results, orient='index')
comparison.index.name = 'Variant'
print(comparison.to_string())

                    Best epoch  Best intent accuracy (%)                         Checkpoint
Variant                                                                                    
Without fallback            32                     85.15    intent_without_fallback_best.pt
With fallback               32                     85.44       intent_with_fallback_best.pt
Full lemmatization          30                     84.95  intent_full_lemmatization_best.pt
